# Open-loop replay — does the policy reproduce TRAINING actions from TRAINING frames?

Decisive test for the **"robot stands still"** issue. It mirrors the RTC `policy_server`'s
exact preprocessing (`make_pre_post_processors` → `predict_action_chunk` → `postprocessor`)
but feeds frames straight from the **dataset** instead of the live robot.

Already ruled out: **OOD start-pose** (reset to manifold, still still) and **broken
normalization** (checkpoint `action`/`state` mean-std are the real dataset stats).

| Result of §2/§3 | Conclusion |
|---|---|
| pred **tracks** ground-truth & **varies** across frames | model + pipeline FINE → fault is the **deploy observation pipeline** (camera images / state scaling at run time) |
| pred **~constant** / does not match GT | **model weights** are the problem (undertrained / collapsed) — norm + preprocessing already cleared |

Run top-to-bottom on the **GPU server** (needs the checkpoint + dataset cache).

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

from lerobot.datasets.lerobot_dataset import LeRobotDataset
from lerobot.policies.factory import get_policy_class, make_pre_post_processors

MODEL = "di-techinnova/smolvla-pouring-0.1"
DATASET = "di-techinnova/so-arm-101-pouring-0.2"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
JOINTS = ["sh_pan", "sh_lift", "elbow", "wr_flex", "wr_roll", "grip"]
print("device:", DEVICE)

## 1. Load policy + the server's pre/post processors

Normalization lives in the **processor pipeline**, not in `policy.named_buffers()` (that only
holds rotary `inv_freq` — expected). This cell loads everything and confirms the stats were
loaded into `pre`/`post` (not identity).

In [ ]:
policy = get_policy_class("smolvla").from_pretrained(MODEL).to(DEVICE).eval()
pre, post = make_pre_post_processors(
    policy.config,
    pretrained_path=MODEL,
    preprocessor_overrides={
        "device_processor": {"device": DEVICE},
        "rename_observations_processor": {"rename_map": {}},  # dataset keys already match the model
    },
    postprocessor_overrides={"device_processor": {"device": DEVICE}},
)
print("preprocessor steps :", [s.__class__.__name__ for s in pre.steps])
print("postprocessor steps:", [s.__class__.__name__ for s in post.steps])

from lerobot.processor.normalize_processor import _NormalizationMixin


def dump_norm(pipeline, label):
    print(f"\n=== {label} ===")
    found = False
    for step in pipeline.steps:
        if isinstance(step, _NormalizationMixin):
            found = True
            stats = getattr(step, "stats", {}) or {}
            print(f"  [{step.__class__.__name__}] norm_map={getattr(step, 'norm_map', None)}")
            for key in ("action", "observation.state"):
                s = stats.get(key)
                if not s:
                    print(f"    {key}: MISSING  <-- would be identity (BROKEN)")
                    continue
                for st in ("mean", "std"):
                    if st in s:
                        print(f"    {key}.{st}: {np.round(np.asarray(s[st]).ravel()[:6], 3)}")
    if not found:
        print("  (no normalization step in this pipeline)")


dump_norm(pre, "PREPROCESSOR")
dump_norm(post, "POSTPROCESSOR")
print("\nExpected action.mean ~ [-25.9, 2.7, 31.2, -32.3, 24.7, 4.4], std ~ [15.8, 31.3, 45.1, 36.6, 34.7, 7.9]")
print("If these match -> normalization is FINE; proceed to section 2.")

## 2. Replay dataset frames spanning episode 0 (approach → pour)

In [ ]:
ds = LeRobotDataset(DATASET)
try:
    f0 = int(ds.episode_data_index["from"][0])
    t0 = int(ds.episode_data_index["to"][0])
except Exception:
    f0, t0 = 0, 450
offsets = [0, 75, 150, 270, 350, 440]
idxs = [f0 + o for o in offsets if f0 + o < t0]

img_keys = [k for k in policy.config.input_features if k.startswith("observation.images.")]
sample0 = ds[f0]
print("model image inputs:", img_keys)
print("dataset provides   :", [k for k in img_keys if k in sample0])
print("replaying global idx:", idxs)

preds, gts, frames = [], [], []
for gi in idxs:
    item = ds[gi]
    obs = {"observation.state": item["observation.state"].unsqueeze(0)}
    for k in img_keys:
        if k in item:  # camera3 is absent in the dataset -> dropped by prepare_images
            img = item[k]
            if img.dtype == torch.uint8:  # match the server's prepare_image: float in [0,1]
                img = img.float() / 255.0
            obs[k] = img.unsqueeze(0)
    obs["task"] = item.get("task", "Pour from orange cup into blue cup.")

    policy.reset()
    processed = pre(obs)
    with torch.no_grad():
        chunk = policy.predict_action_chunk(processed)  # (B, chunk, dim), normalized
    if chunk.ndim != 3:
        chunk = chunk.unsqueeze(0)
    pred = post(chunk[:, 0, :]).detach().cpu().numpy().ravel()[:6]  # unnormalized first action
    gt = item["action"].detach().cpu().numpy().ravel()[:6]
    fi = int(item["frame_index"].item()) if "frame_index" in item else gi - f0
    preds.append(pred)
    gts.append(gt)
    frames.append(fi)
    print(f"frame {fi:3d} | pred {np.round(pred, 1)} | gt {np.round(gt, 1)} | |err| {np.round(np.abs(pred - gt), 1)}")

preds = np.stack(preds)
gts = np.stack(gts)
frames = np.array(frames)

## 3. Verdict

In [ ]:
mae = float(np.abs(preds - gts).mean())
pred_var = preds.std(0)
gt_var = gts.std(0)
print(f"mean |pred - GT|             : {mae:.2f} deg")
print(f"pred variation across frames : {np.round(pred_var, 1)}")
print(f"GT   variation across frames : {np.round(gt_var, 1)}")
print()
reproduces = mae < 8 and pred_var.mean() > 0.4 * gt_var.mean()
if reproduces:
    print("=> MODEL REPRODUCES TRAINING DATA.")
    print("   Fault is the DEPLOY observation pipeline: compare the live camera images and")
    print("   state scaling against the dataset frames (color space, camera assignment, units).")
else:
    print("=> MODEL DOES NOT REPRODUCE (output near-constant or off).")
    print("   Norm + preprocessing already cleared (section 1), so suspect the WEIGHTS:")
    print("   check actual training steps / final loss, and that from_pretrained loaded them.")

In [ ]:
fig, axs = plt.subplots(2, 3, figsize=(13, 6))
for j, ax in enumerate(axs.ravel()):
    ax.plot(frames, gts[:, j], "o-", label="ground-truth", color="tab:green")
    ax.plot(frames, preds[:, j], "x--", label="pred", color="tab:red")
    ax.set_title(JOINTS[j])
    ax.set_xlabel("frame")
    ax.grid(alpha=0.3)
    if j == 0:
        ax.legend()
fig.suptitle("Open-loop: predicted vs ground-truth action per joint (episode 0)")
plt.tight_layout()
plt.show()

## 4. How to read this

- **Lines overlap & both move** (pred follows GT from approach to pour) → model + pipeline are
  correct. The standstill is a **deploy-time observation** problem: the live camera images or the
  reported `observation.state` differ from training (color/space/resolution/crop, camera
  assignment, or a state scale/units mismatch). Next: dump a live deploy observation and diff it
  against a dataset frame.

- **Red (pred) flat / far from green** → the model itself is not conditioning on inputs. Since
  normalization + preprocessing are already cleared, suspect the **weights**: undertrained or a
  mis-saved checkpoint produce exactly this near-constant output.